Databricks allows us to write 
1.  PySpark DataFrame API- spark code (python)
2.  Spark SQL- sql code

all at a single place without bothering about infrastructure or installation thus allowing seamless cross platform integration

Databricks automatically provides a SparkSession (spark) when you start a notebook.


In [0]:
print("sprak version =",spark.version)
print("spark configuration=",spark.conf.getAll) # All configuration settings


### Version 1: PySpark DataFrame API

You write code in Python using functions like .select(), .filter(), .groupBy().
More programmatic & flexible (easy to integrate with Python logic, UDFs).

In [0]:
#catalog==>Add Data==> Upload to Volume ==> Browse file ==> set destination path/Volumes ==> copy file path
# OR another way is Home==>bring in data


# Python API
df = spark.read.option("header", True).csv("/Volumes/workspace/default/workspace_volume/sample_databricks.csv")

# select columns
df.select("name", "age").show(5)

# filter rows
df.filter(df.age > 30).show(50)

### ✅ Version 2: Spark SQL

You register a DataFrame as a temporary view/table.

Then run SQL queries (just like querying a database).

Great for SQL-style analytics or when team members know SQL better.

In [0]:
# Register DataFrame as a view
df.createOrReplaceTempView("people")

# Run SQL query when you are in a python cell
spark.sql("SELECT name, age FROM people WHERE age > 30").show()

# or if you are in a SQL cell
# %sql
# SELECT name, age
# FROM people
# WHERE age > 30


Databricks notebooks default to Python, so it interprets SQL as Python code
Use %sql magic in your notebook

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW people
USING csv
OPTIONS (
  path "/Volumes/workspace/default/workspace_volume/sample_databricks.csv",
  header "true",
  inferSchema "true"
);


In [0]:
%sql
select * from people

###Dir vs Help

dir(spark) will return all methods and attributes available in this class

but if further documentation is required for any method you go for help(spark.MethodName)

In [0]:
import pprint
pprint.pprint(dir(spark))

In [0]:
pprint.pprint(help(spark.readStream))

In [0]:
people_path = "/Volumes/workspace/default/workspace_volume/sample_databricks.csv"
df_people = spark.read.option("header", True).option("inferSchema", True).csv(people_path)
##All columns will be read as strings unless you define a schema manually.-infer schema
df_people.printSchema()

## Important Options
header=	First row has column names

inferSchema=	Automatically detect data types

delimiter or sep=	If file uses `

quote=	Handle quoted strings

escape=	Escape special characters

nullValue=	Define what counts as null

mode=	How to handle bad records

multiLine = Needed when JSON spans multiple line

columnNameOfCorruptRecord= Store bad JSON

dropFieldIfAllNull =Remove null-only fields

mergeSchema=	Handle schema evolution

pathGlobFilter=	Filter specific files

recursiveFileLookup=	Read nested directories



###To Navigate Storage in Databricks

A) DBFS (Databricks File System)

B) Mount Points (/mnt) [external storage.]

C) Unity Catalog Volumes (Modern Way)

Databricks clusters just compute.
Storage lives in cloud object storage.




In [0]:
# DBFS is a layer on top of cloud storage.
# You can access it like this:
display(dbutils.fs.ls("/"))

# or use %fs magic command


DBFS is virtual file system interface over cloud object storage.

%fs is a magic command used inside notebooks to interact with the Databricks File System (DBFS).- shorthand for calling dbutils.fs
allows to run run file system operations directly in a notebook cell

all these commands are same as linux

However,Databricks API → DBFS layer → S3/ADLS → Cloud storage
So:

Linux ls → reads local disk

%fs ls → Internally: ==>Calls DBFS API
=>DBFS maps to S3 bucket
=>Lists objects in cloud storage
==>No local disk involved.

they look like Linux commands, and that’s intentional.
But they are not actually Linux commands.they are wrappers around Databricks file APIs that interact with cloud object storage.

###Essential DBFS Commands
DBFS (Databricks File System) is the persistent file layer. You access it via the dbutils.fs utility in notebooks.



In [0]:
# LIST — see what's in a directory
dbutils.fs.ls("/")
dbutils.fs.ls("/FileStore")
dbutils.fs.ls("/user/hive/warehouse")

# Pretty display in notebook
display(dbutils.fs.ls("/FileStore"))

# CREATE a directory
dbutils.fs.mkdirs("/FileStore/my_project/raw")

# COPY a file or directory
dbutils.fs.cp("/FileStore/source.csv", "/FileStore/backup/source.csv")
dbutils.fs.cp("/FileStore/folder1", "/FileStore/folder2", recurse=True)

# MOVE (rename) a file or directory
dbutils.fs.mv("/FileStore/old_name.csv", "/FileStore/new_name.csv")

# REMOVE a file
dbutils.fs.rm("/FileStore/temp.csv")

# REMOVE a directory recursively
dbutils.fs.rm("/FileStore/old_folder", recurse=True)

# READ a small file's contents
content = dbutils.fs.head("/FileStore/sample.csv", maxBytes=2048)
print(content)

# WRITE a small text file
dbutils.fs.put("/FileStore/notes.txt", "Hello world", overwrite=True)

###%fs Magic Command (SQL-like Shorthand)
In notebook cells, you can use %fs instead of dbutils.fs:

 %fs ls /FileStore

 %fs ls /user/hive/warehouse

 %fs mkdirs /FileStore/new_dir

 %fs rm -r /FileStore/temp_dir
 
 %fs head /FileStore/sample.csv

## Common DBFS Paths to Know

| Path | What's there |
|---|---|
| `/FileStore/` | User uploads, downloadable files |
| `/user/hive/warehouse/` | Default location for managed tables |
| `/databricks-datasets/` | Built-in sample datasets (read-only) |
| `/tmp/` | Temporary files (still in DBFS, but conventionally short-lived) |
| `/mnt/` | Mounted external storage (S3, ADLS, etc.) |

##Linux in Data bricks

we can use Linux commands. The trick is understanding which filesystem they're hitting.
The %sh Magic Command
%sh runs a shell command on the driver node's local filesystem — NOT on DBFS directly.


If you run:

%sh <command>

Now this is real Linux.
It runs on:
Driver node==>
Inside cluster VM
That is actual OS filesystem.

DBFS is also mounted as a regular Linux path at /dbfs/. So:

| Access from | DBFS path syntax |
|---|---|
| dbutils.fs or Spark | dbfs:/FileStore/file.csv |
|Linux shell (%sh) or Python open()| /dbfs/FileStore/file.csv|


Why This Difference Matters

Imagine:

You delete a file with:
%fs rm /mnt/data.csv

You are deleting from:
→ S3 / ADLS
Not local disk.
That can impact production data.

It runs on the driver node VM of your cluster.

So:

It is real Linux
But inside the cluster

Not your laptop

Not production server

Files created via %sh:
Live on driver local disk and
Are temporary they
Disappear when cluster restarts

So this is NOT persistent storage.

In [0]:
# %sh

# Check disk space
df -h

# Check directory size
du -sh /dbfs/FileStore/my_data

# Count lines in a file
wc -l /dbfs/FileStore/sample.csv

# Search inside files
grep -r "error" /dbfs/FileStore/logs/

# View file head/tail
head -20 /dbfs/FileStore/sample.csv
tail -20 /dbfs/FileStore/sample.csv

# Find files
find /dbfs/FileStore -name "*.parquet" -size +10M

# Compress / decompress
gzip /dbfs/FileStore/big_file.csv
gunzip /dbfs/FileStore/big_file.csv.gz
tar -czvf archive.tar.gz folder/

# Download files from the internet to the driver
wget https://example.com/data.csv -O /tmp/data.csv
curl -o /tmp/data.json https://api.example.com/data

# Then move it to DBFS for persistence
# Switch back to Python:
dbutils.fs.cp("file:/tmp/data.csv", "dbfs:/FileStore/data.csv")

###Installing Linux Packages

In [0]:
# %sh
# apt-get install -y tree
# tree /dbfs/FileStore -L 2

# Install Python packages (use %pip instead, it's better)
# pip install requests

#For Python packages, prefer %pip install — it handles cluster-wide installation properly:
# %pip install pandas-profiling
# %pip install some-library

In [0]:
###Running Multi-Line Shell Scripts

In [0]:
# %sh
# cd /dbfs/FileStore
# for f in *.csv; do
#   echo "Processing $f"
#   wc -l "$f"
# done